In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, sum as _sum, mean, stddev, countDistinct, lit
from pyspark.sql.types import NumericType, StructType, StructField, StringType, DoubleType, IntegerType
from typing import List, Dict, Any

class DataQualityChecker:
    """Класс для проверки качества данных"""
    
    def __init__(self, df: DataFrame):
        self.df = df
        self.total_rows = df.count()
    
    def check_nulls(self, columns: List[str] = None) -> DataFrame:
        """
        Проверка null значений в указанных колонках
        """
        if columns is None:
            columns = self.df.columns
        
        null_stats = []
        for col_name in columns:
            null_count = self.df.filter(col(col_name).isNull()).count()
            null_percent = (null_count / self.total_rows) * 100 if self.total_rows > 0 else 0
            # Приводим к float
            null_stats.append((
                str(col_name), 
                float(null_count), 
                float(round(null_percent, 2))
            ))
        
        # Создаем DataFrame со статистикой
        schema = StructType([
            StructField("column", StringType(), True),
            StructField("null_count", DoubleType(), True),
            StructField("null_percent", DoubleType(), True)
        ])
        
        return self.df.sparkSession.createDataFrame(null_stats, schema)
    
    def check_duplicates(self, subset: List[str] = None) -> DataFrame:
        """
        Проверка дубликатов
        """
        if subset:
            duplicate_count = self.df.groupBy(subset).count() \
                .filter(col('count') > 1).count()
        else:
            duplicate_count = self.df.count() - self.df.distinct().count()
        
        duplicate_percent = (duplicate_count / self.total_rows) * 100 if self.total_rows > 0 else 0
        
        # Создаем DataFrame с результатом
        result = [(
            float(self.total_rows),
            float(duplicate_count),
            float(round(duplicate_percent, 2))
        )]
        
        schema = StructType([
            StructField("total_rows", DoubleType(), True),
            StructField("duplicate_count", DoubleType(), True),
            StructField("duplicate_percent", DoubleType(), True)
        ])
        
        return self.df.sparkSession.createDataFrame(result, schema)
    
    def check_data_types(self) -> DataFrame:
        """
        Проверка типов данных
        """
        type_stats = []
        for field in self.df.schema.fields:
            type_stats.append((str(field.name), str(field.dataType), str(field.nullable)))
        
        schema = StructType([
            StructField("column", StringType(), True),
            StructField("data_type", StringType(), True),
            StructField("nullable", StringType(), True)
        ])
        
        return self.df.sparkSession.createDataFrame(type_stats, schema)
    
    def check_unique_constraint(self, columns: List[str]) -> DataFrame:
        """
        Проверка уникальности составного ключа
        """
        total_unique = self.df.select(columns).distinct().count()
        is_unique = total_unique == self.total_rows
        duplicate_count = self.total_rows - total_unique
        
        result = [(
            ','.join(columns),
            str(is_unique),
            float(total_unique),
            float(self.total_rows),
            float(duplicate_count)
        )]
        
        schema = StructType([
            StructField("columns", StringType(), True),
            StructField("is_unique", StringType(), True),
            StructField("unique_count", DoubleType(), True),
            StructField("total_count", DoubleType(), True),
            StructField("duplicate_count", DoubleType(), True)
        ])
        
        return self.df.sparkSession.createDataFrame(result, schema)

In [2]:
def get_column_statistics(df: DataFrame, columns: List[str] = None) -> DataFrame:
    """
    Получение статистики по колонкам
    """
    if columns is None:
        columns = df.columns
    
    stats_list = []
    for col_name in columns:
        col_type = df.schema[col_name].dataType
        
        # Базовая статистика
        null_count = df.filter(col(col_name).isNull()).count()
        distinct_count = df.select(col_name).distinct().count()
        
        # Числовая статистика
        if isinstance(col_type, NumericType):
            stats = df.select(
                mean(col_name).alias('mean'),
                stddev(col_name).alias('stddev')
            ).collect()[0]
            
            nan_count = df.filter(isnan(col(col_name))).count()
            
            stat_row = (
                str(col_name),
                str(col_type),
                float(df.count()),
                float(null_count),
                float(distinct_count),
                float(stats['mean']) if stats['mean'] is not None else 0.0,
                float(stats['stddev']) if stats['stddev'] is not None else 0.0,
                float(nan_count)
            )
        else:
            stat_row = (
                str(col_name),
                str(col_type),
                float(df.count()),
                float(null_count),
                float(distinct_count),
                None,
                None,
                float(0)
            )
        
        stats_list.append(stat_row)
    
    schema = StructType([
        StructField("column", StringType(), True),
        StructField("type", StringType(), True),
        StructField("total_count", DoubleType(), True),
        StructField("null_count", DoubleType(), True),
        StructField("distinct_count", DoubleType(), True),
        StructField("mean", DoubleType(), True),
        StructField("stddev", DoubleType(), True),
        StructField("nan_count", DoubleType(), True)
    ])
    
    return df.sparkSession.createDataFrame(stats_list, schema)

In [3]:
class DataValidator:
    """Класс для валидации форматов данных"""
    
    @staticmethod
    def validate_email(df: DataFrame, email_col: str) -> DataFrame:
        """
        Валидация email адресов
        """
        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        
        return df.withColumn(
            'is_valid_email',
            when(col(email_col).rlike(email_pattern), True).otherwise(False)
        )
    
    @staticmethod
    def validate_phone(df: DataFrame, phone_col: str, country: str = 'ru') -> DataFrame:
        """
        Валидация телефонных номеров
        """
        patterns = {
            'ru': r'^(\+7|8)?[\s\-]?\(?[0-9]{3}\)?[\s\-]?[0-9]{3}[\s\-]?[0-9]{2}[\s\-]?[0-9]{2}$',
            'us': r'^(\+1)?[\s\-]?\(?[0-9]{3}\)?[\s\-]?[0-9]{3}[\s\-]?[0-9]{4}$'
        }
        
        pattern = patterns.get(country, patterns['ru'])
        
        return df.withColumn(
            f'is_valid_{phone_col}',
            when(col(phone_col).rlike(pattern), True).otherwise(False)
        )
    
    @staticmethod
    def validate_date(df: DataFrame, date_col: str, date_format: str = 'yyyy-MM-dd') -> DataFrame:
        """
        Валидация дат
        """
        from pyspark.sql.functions import to_date, isnull
        
        df_validated = df.withColumn('_parsed_date', to_date(col(date_col), date_format))
        
        return df_validated.withColumn(
            f'is_valid_{date_col}',
            when(isnull(col('_parsed_date')), False).otherwise(True)
        ).drop('_parsed_date')
    
    @staticmethod
    def validate_range(df: DataFrame, column: str, min_val: Any, max_val: Any) -> DataFrame:
        """
        Валидация диапазона значений
        """
        return df.withColumn(
            f'{column}_in_range',
            when((col(column) >= min_val) & (col(column) <= max_val), True).otherwise(False)
        )
    
    @staticmethod
    def get_validation_summary(df: DataFrame, validation_col: str) -> DataFrame:
        """
        Получение сводки по валидации
        """
        total = float(df.count())
        valid_count = float(df.filter(col(validation_col) == True).count())
        invalid_count = total - valid_count
        valid_percent = (valid_count / total) * 100 if total > 0 else 0.0
        
        result = [(
            str(validation_col),
            total,
            valid_count,
            invalid_count,
            float(round(valid_percent, 2))
        )]
        
        schema = StructType([
            StructField("validation_column", StringType(), True),
            StructField("total", DoubleType(), True),
            StructField("valid", DoubleType(), True),
            StructField("invalid", DoubleType(), True),
            StructField("valid_percent", DoubleType(), True)
        ])
        
        return df.sparkSession.createDataFrame(result, schema)

In [4]:
def check_business_rules(df: DataFrame, rules: Dict[str, str]) -> DataFrame:
    """
    Проверка бизнес-правил
    rules: {'rule_name': 'sql_condition'}
    """
    results = []
    total = float(df.count())
    
    for rule_name, condition in rules.items():
        # Подсчет нарушений
        violations = float(df.filter(condition).count())
        pass_rate = ((total - violations) / total) * 100 if total > 0 else 0.0
        
        results.append((
            str(rule_name), 
            str(condition), 
            violations, 
            total, 
            float(round(pass_rate, 2))
        ))
    
    schema = StructType([
        StructField("rule_name", StringType(), True),
        StructField("condition", StringType(), True),
        StructField("violations", DoubleType(), True),
        StructField("total", DoubleType(), True),
        StructField("pass_rate", DoubleType(), True)
    ])
    
    return df.sparkSession.createDataFrame(results, schema)


def check_foreign_key(df: DataFrame, 
                      fk_column: str, 
                      reference_df: DataFrame, 
                      reference_column: str) -> tuple:
    """
    Проверка ссылочной целостности
    """
    # Находим значения, отсутствующие в справочнике
    missing_values_df = df.select(fk_column).distinct() \
        .join(reference_df.select(reference_column).distinct(),
              df[fk_column] == reference_df[reference_column],
              'left_anti')
    
    missing_count = float(missing_values_df.count())
    total_distinct = float(df.select(fk_column).distinct().count())
    integrity_rate = ((total_distinct - missing_count) / total_distinct) * 100 if total_distinct > 0 else 0.0
    
    # Создаем результат
    result = [(
        str(fk_column),
        total_distinct,
        missing_count,
        float(round(integrity_rate, 2))
    )]
    
    schema = StructType([
        StructField("fk_column", StringType(), True),
        StructField("total_distinct_values", DoubleType(), True),
        StructField("missing_in_reference", DoubleType(), True),
        StructField("integrity_rate", DoubleType(), True)
    ])
    
    summary = df.sparkSession.createDataFrame(result, schema)
    
    return summary, missing_values_df

In [5]:
def generate_dq_report(df: DataFrame, 
                       pk_columns: List[str] = None,
                       email_columns: List[str] = None,
                       phone_columns: List[str] = None,
                       date_columns: List[str] = None) -> Dict[str, DataFrame]:
    """
    Генерация полного отчета о качестве данных
    Возвращает словарь с DataFrames
    """
    dq_checker = DataQualityChecker(df)
    validator = DataValidator()
    
    report = {}
    
    # Базовая статистика
    basic_stats = [(
        df.count(),
        len(df.columns),
        ', '.join(df.columns)
    )]
    
    schema_basic = StructType([
        StructField("total_rows", DoubleType(), True),
        StructField("total_columns", DoubleType(), True),
        StructField("columns", StringType(), True)
    ])
    
    report['basic_stats'] = df.sparkSession.createDataFrame(basic_stats, schema_basic)
    
    # Null анализ
    report['null_analysis'] = dq_checker.check_nulls()
    
    # Duplicate анализ
    report['duplicate_analysis'] = dq_checker.check_duplicates(pk_columns)
    
    # Типы данных
    report['data_types'] = dq_checker.check_data_types()
    
    # Статистика по колонкам
    report['column_statistics'] = get_column_statistics(df)
    
    # Валидация email
    if email_columns:
        for email_col in email_columns:
            if email_col in df.columns:
                validated = validator.validate_email(df, email_col)
                report[f'{email_col}_validation'] = validator.get_validation_summary(
                    validated, 'is_valid_email'
                )
    
    # Валидация телефонов
    if phone_columns:
        for phone_col in phone_columns:
            if phone_col in df.columns:
                validated = validator.validate_phone(df, phone_col)
                report[f'{phone_col}_validation'] = validator.get_validation_summary(
                    validated, f'is_valid_{phone_col}'
                )
    
    # Валидация дат
    if date_columns:
        for date_col in date_columns:
            if date_col in df.columns:
                validated = validator.validate_date(df, date_col)
                report[f'{date_col}_validation'] = validator.get_validation_summary(
                    validated, f'is_valid_{date_col}'
                )
    
    return report

# пример

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder.appName("DQ").getOrCreate()

# Создаем тестовый DataFrame
data = [
    (1, "user1@example.com", "+79123456789", "2024-01-01", 100),
    (2, "invalid-email", "+123456789", "2024-01-02", 200),
    (3, None, "89123456789", "invalid-date", 300),
    (4, "user4@example.com", None, "2024-01-04", None),
    (4, "user4@example.com", "+79123456789", "2024-01-04", 400)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("date", StringType(), True),
    StructField("value", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)

# Создаем объект для проверки
dq = DataQualityChecker(df)

# Проверяем null значения
print("Null values:")
dq.check_nulls().show()

# Проверяем дубликаты
print("\nDuplicates:")
dq.check_duplicates(subset=['id']).show()

# Статистика по колонкам
print("\nColumn statistics:")
get_column_statistics(df).show(truncate=False)

# Валидация email
validator = DataValidator()
validated_df = validator.validate_email(df, 'email')
print("\nEmail validation:")
validated_df.select('email', 'is_valid_email').show()

# Сводка по валидации email
print("\nEmail validation summary:")
validator.get_validation_summary(validated_df, 'is_valid_email').show()

# Бизнес-правила
rules = {
    "positive_value": "value < 0",
    "null_value": "value is null"
}
print("\nBusiness rules check:")
check_business_rules(df, rules).show()

Null values:
+------+----------+------------+
|column|null_count|null_percent|
+------+----------+------------+
|    id|       0.0|         0.0|
| email|       1.0|        20.0|
| phone|       1.0|        20.0|
|  date|       0.0|         0.0|
| value|       1.0|        20.0|
+------+----------+------------+


Duplicates:
+----------+---------------+-----------------+
|total_rows|duplicate_count|duplicate_percent|
+----------+---------------+-----------------+
|       5.0|            1.0|             20.0|
+----------+---------------+-----------------+


Column statistics:
+------+-------------+-----------+----------+--------------+-----+------------------+---------+
|column|type         |total_count|null_count|distinct_count|mean |stddev            |nan_count|
+------+-------------+-----------+----------+--------------+-----+------------------+---------+
|id    |IntegerType()|5.0        |0.0       |4.0           |2.8  |1.3038404810405297|0.0      |
|email |StringType() |5.0        |1.

NameError: name 'DataValidator' is not defined